# 3D Reconstruction from 2D Images

## Phase 1: Data Loading and Verification

### 1.1 Setup Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Clone/Update Repo and Install Dependencies

In [ ]:
import os

repo_path = '/content/bohboh'

if not os.path.exists(repo_path):
  !git clone https://github.com/hazempgm/bohboh.git
  %cd {repo_path}
else:
  %cd {repo_path}
  !git pull

!git checkout version_3

# Install all required libraries
!pip install tifffile matplotlib scikit-image tqdm

### 1.3 Define Paths and Add to System Path

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add the src directory to the Python path
sys.path.append(os.path.abspath('src'))

# Define the path to the image data
image_dir = '/content/drive/MyDrive/bahbouh/data_bohboh/Images'
output_dir = '/content/drive/MyDrive/bahbouh/' # For saving results

print(f"Source code path: {os.path.abspath('src')}")
print(f"Data path: {image_dir}")

### 1.4 Load Images

In [ ]:
from data_loader import load_tiff_images

image_stack = load_tiff_images(image_dir)

if image_stack.size > 0:
    print(f"\nImage stack loaded successfully.")
    print(f"Shape of the stack: {image_stack.shape}")

---

## Phase 2: Single Slice Reconstruction (Verification)

In [ ]:
from reconstruction import reconstruct_slice

# We will only run this on a small part of the data to verify
if image_stack.size > 0:
    slice_height_index = image_stack.shape[1] // 2
    sinogram = image_stack[:, slice_height_index, :]
    theta = np.linspace(0., 180., image_stack.shape[0], endpoint=False)

    reconstructed_test_slice = reconstruct_slice(sinogram, theta)

    # Visualize the result
    plt.figure(figsize=(6, 6))
    plt.imshow(reconstructed_test_slice, cmap='gray')
    plt.title('Test: Reconstructed 2D Cross-Section')
    plt.show()

---

## Phase 3: Full 3D Volume Reconstruction

This is a computationally expensive process. We will first downsample the data to a more manageable size.

### 3.1 Downsample the Projection Stack

In [ ]:
from skimage.transform import resize

# Define the new, smaller size for the projections
# New shape will be (num_angles, new_height, new_width)
new_shape = (image_stack.shape[0], 512, 512)

print(f"Original shape: {image_stack.shape}")
print(f"Downsampling to: {new_shape}...")

image_stack_small = resize(image_stack, new_shape, anti_aliasing=True)

print("Downsampling complete.")
print(f"New shape: {image_stack_small.shape}")

### 3.2 Reconstruct the Full Volume

In [ ]:
from reconstruction import reconstruct_full_volume

# This will take some time to run. The progress bar will show the status.
reconstructed_volume = reconstruct_full_volume(image_stack_small)

print(f"\nFinal reconstructed volume shape: {reconstructed_volume.shape}")

### 3.3 Save the Reconstructed Volume

In [ ]:
output_path = os.path.join(output_dir, 'reconstructed_volume_512.npy')
print(f"Saving reconstructed volume to: {output_path}")

np.save(output_path, reconstructed_volume)

print("File saved successfully.")